In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("anasmohammedtahir/covidqu")

print("Path to dataset files:", path)

100%|██████████| 1.15G/1.15G [00:11<00:00, 109MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/anasmohammedtahir/covidqu/versions/7


In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tensorflow.keras.models import load_model
import numpy as np
import os

In [ ]:
# For torch models
transform_pytorch = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# For inception
transform_incep = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),  # no Normalize here
])



In [ ]:
class CustomCOVIDDataset(Dataset):
    def __init__(self, root_dir, transform=None, classes=["COVID-19", "Non-COVID", "Normal"]):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = classes
        self.samples = []
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}

        for cls in classes:
            img_dir = os.path.join(root_dir, cls, "images")
            if not os.path.exists(img_dir):
                continue
            for img_name in os.listdir(img_dir):
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    img_path = os.path.join(img_dir, img_name)
                    label = self.class_to_idx[cls]
                    self.samples.append((img_path, label))

        print(f"✅ Loaded {len(self.samples)} images from {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label


In [ ]:
TEST_DIR = "/root/.cache/kagglehub/datasets/anasmohammedtahir/covidqu/versions/7/Lung Segmentation Data/Lung Segmentation Data/Test"

In [ ]:
test_dataset_224 = CustomCOVIDDataset(root_dir=TEST_DIR, transform=transform_pytorch)
test_loader_224 = DataLoader(test_dataset_224, batch_size=16, shuffle=False)


✅ Loaded 6788 images from /root/.cache/kagglehub/datasets/anasmohammedtahir/covidqu/versions/7/Lung Segmentation Data/Lung Segmentation Data/Test


In [ ]:
test_dataset_296 = CustomCOVIDDataset(root_dir=TEST_DIR, transform=transform_incep)
test_loader_296 = DataLoader(test_dataset_296, batch_size=16, shuffle=False)

✅ Loaded 6788 images from /root/.cache/kagglehub/datasets/anasmohammedtahir/covidqu/versions/7/Lung Segmentation Data/Lung Segmentation Data/Test


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
DRIVE_PATH = "/content/drive/MyDrive/Abnormality Detection Model"

EFFICIENTNET_PATH = os.path.join(DRIVE_PATH, "covidqu_model.pth")
RESNET_PATH       = os.path.join(DRIVE_PATH, "lung_segementation.pth")
DENSENET_PATH     = os.path.join(DRIVE_PATH, "densenet.pth")
VGG_PATH          = os.path.join(DRIVE_PATH, "vgg16_covid_final_best_model.pth")
INCEPTION_PATH    = os.path.join(DRIVE_PATH, "inceptionresnet.h5")



In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

✅ Using device: cpu


In [ ]:
pip install efficientnet_pytorch

  Preparing metadata (setup.py) ... done
  Created wheel for efficientnet_pytorch: filename=efficientnet_pytorch-0.7.1-py3-none-any.whl size=16426 sha256=e16ea2b34ffe1470dbdc12447b78ea66bc56e01aac1ca9153a3e63faeddd3ddc
  Stored in directory: /root/.cache/pip/wheels/9c/3f/43/e6271c7026fe08c185da2be23c98c8e87477d3db63f41f32ad
Successfully built efficientnet_pytorch


In [ ]:
import torch
import torch.nn as nn
from torchvision import models
from efficientnet_pytorch import EfficientNet

def build_torch_models(device):
    # 1️⃣ EfficientNetB0
    efficientnet = EfficientNet.from_name('efficientnet-b0')
    efficientnet._fc = nn.Linear(efficientnet._fc.in_features, 3)
    efficientnet.load_state_dict(torch.load(EFFICIENTNET_PATH, map_location=device))
    efficientnet.to(device)
    efficientnet.eval()

    # 2️⃣ ResNet18
    resnet = models.resnet18(weights=None)
    resnet.fc = nn.Linear(resnet.fc.in_features, 3)
    resnet.load_state_dict(torch.load(RESNET_PATH, map_location=device))
    resnet.to(device)
    resnet.eval()

    # 3️⃣ DenseNet121
    densenet = models.densenet121(weights=None)
    densenet.classifier = nn.Linear(densenet.classifier.in_features, 3)
    densenet.load_state_dict(torch.load(DENSENET_PATH, map_location=device))
    densenet.to(device)
    densenet.eval()

    vgg = models.vgg16(weights=None)

    # Rebuild classifier to match checkpoint
    vgg.classifier = nn.Sequential(
        nn.Linear(25088, 4096),
        nn.ReLU(inplace=True),
        nn.Dropout(),
        nn.Linear(4096, 1024),   # matches checkpoint
        nn.ReLU(inplace=True),
        nn.Dropout(),
        nn.Linear(1024, 3)       # matches checkpoint (3 classes)
    )

    # Load checkpoint
    checkpoint = torch.load(VGG_PATH, map_location=device)
    state_dict = checkpoint['model_state_dict']
    state_dict = {k.replace('vgg16.', ''): v for k, v in state_dict.items()}
    vgg.load_state_dict(state_dict)

    vgg.to(device)
    vgg.eval()



    print("✅ All PyTorch models rebuilt and weights loaded.")
    return efficientnet, resnet, densenet, vgg

efficientnet, resnet, densenet, vgg = build_torch_models(device)


✅ All PyTorch models rebuilt and weights loaded.


In [ ]:
from tensorflow.keras.applications.inception_resnet_v2 import InceptionResNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
import tensorflow as tf

def load_inception_model(model_path, num_classes=3):
    base = InceptionResNetV2(weights='imagenet', include_top=False, input_shape=(299,299,3))
    base.trainable = False

    x = GlobalAveragePooling2D()(base.output)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.25)(x)
    out = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base.input, outputs=out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    if os.path.exists(model_path):
        model.load_weights(model_path)
        print(f"✅ Loaded InceptionResNetV2 weights from {model_path}")
    else:
        print("⚠️ Model path not found, using untrained weights!")

    return model

inception_model = load_inception_model(INCEPTION_PATH)

219055592/219055592 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
✅ Loaded InceptionResNetV2 weights from /content/drive/MyDrive/Abnormality Detection Model/inceptionresnet.h5


In [ ]:
def torch_prob(model,x):
  model.eval()
  with torch.no_grad():
    logits = model(x)
    probs = F.softmax(logits,dim=1)

  return probs


from tensorflow.keras.applications.inception_resnet_v2 import preprocess_input

def keras_prob(model, x):
    x_np = x.permute(0, 2, 3, 1).cpu().numpy() * 255.0  # convert back to 0–255
    x_np = preprocess_input(x_np)  # scale to [-1, 1]
    probs = model.predict(x_np, verbose=0)
    return torch.tensor(probs, device=device)


In [ ]:
models = [efficientnet, inception_model, resnet, densenet, vgg]
weights = [0.3, 0.25, 0.15, 0.2, 0.1]

In [ ]:
def ensemble_predict(models, weights, x_pytorch, x_incep):
    total_probs = None
    total_weight = sum(weights)

    for model, weight in zip(models, weights):
        if isinstance(model, torch.nn.Module):
            probs = torch_prob(model, x_pytorch)
        else:
            probs = keras_prob(model, x_incep)

        probs = probs / probs.sum(dim=1, keepdim=True)  # normalize per sample
        if total_probs is None:
            total_probs = weight * probs
        else:
            total_probs += weight * probs

    total_probs /= total_weight
    preds = torch.argmax(total_probs, dim=1)
    return preds


In [ ]:
torch_models = [efficientnet, resnet, densenet, vgg]
keras_model = inception_model

all_preds = []
all_labels = []

# Check the contents of the test directory
print(f"Contents of {TEST_DIR}: {os.listdir(TEST_DIR)}")

# Iterate over both loaders
for (x_pytorch, labels_pytorch), (x_incep, _) in zip(test_loader_224, test_loader_296):
    x_pytorch = x_pytorch.to(device)
    labels_pytorch = labels_pytorch.to(device)

    # Ensemble prediction
    preds = ensemble_predict(torch_models + [keras_model],
                             weights=weights,  # adjust weights if needed
                             x_pytorch=x_pytorch,
                             x_incep=x_incep.to(device))

    all_preds.append(preds.cpu())
    all_labels.append(labels_pytorch.cpu())

# Concatenate all predictions and labels
all_preds = torch.cat(all_preds)
all_labels = torch.cat(all_labels)

print("✅ Ensemble prediction done.")

Contents of /root/.cache/kagglehub/datasets/anasmohammedtahir/covidqu/versions/7/Lung Segmentation Data/Lung Segmentation Data/Test: ['Normal', 'Non-COVID', 'COVID-19']


KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Accuracy
acc = accuracy_score(all_labels, all_preds)
print(f"Ensemble Accuracy: {acc:.4f}")

# Confusion Matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=test_dataset_224.classes, yticklabels=test_dataset_224.classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# Classification report
print(classification_report(all_labels, all_preds, target_names=test_dataset_224.classes))
